In [4]:
# transfromer 하이퍼파라미터 및 각종 환경 설정
import torch
import torch.nn as nn 
import torch.optim as optim
import torch.nn.functional as F 

d_model = 512  # 모델 임베딩 차원
num_layers = 6  # Encoder / Decoder 블록(레이어) 수
num_heads = 8  # Multi-Head Attention 헤드 수
d_k = d_model // num_heads  # 헤드당 Q, K, V 차원 수 (512//8 = 64)
d_ff = 2048  # Position-wise FFN의 은닉 차원
drop_out_rate = 0.1  # Dropout 비율(과적합 완화)

src_vocab_size = 10000  # 원문 사전 크기
trg_vocab_size = 10000  # 번역문 사전 크기

batch_size = 64
seq_len = 128  # 최대 시퀀스 길이 (패딩 처리)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
# Transformer 모델 뼈대 정의 (Embedding -> Encoder -> Decoder -> Output)

# 토큰 임베딩에 위치 정보(순서)를 더해주는 Positional Encoding 모듈
class PositionalEncoding(nn.Module):
    pass

# 소스 입력을 인코딩해 문맥 표현을 만드는 Enoder
class Encoder(nn.Module):
    pass

# 타겟 입력과 encoder outputs를 이용해 디코딩 출력 시퀀스 생성하는 Decoder
class Decoder(nn.Module):
    pass

In [6]:
# Transformer 전체 흐름 : 임베딩 -> 포지셔널 -> 인코더 -> 디코더 -> 출력층
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model):
        super().__init__()  # 모듈 초기화
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)  # 소스(원문) 토큰 ID 받아서 ID를 임베딩(d_model) 작업
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)  # 타겟(번역문) 토큰 ID 받아서 ID를 임베딩(d_model) 작업
        self.positional_encoding = PositionalEncoding()  # 위치 정보 인코딩 모듈
        self.encoder = Encoder()  # 인코더
        self.decoder = Decoder()  # 디코더
        self.output_layer = nn.Linear(d_model, trg_vocab_size)  # 디코더 출력(d_model) -> 어휘 로짓(vocab)
        self.softmax = nn.LogSoftmax(dim=1)  # 마지막 차원 기준 로그 확률 변환

    # 순전파 : src_inputs, trg_inputs를 받아서 번역문 토큰을 로그확률 분포로 출력
    def forward(self, src_inputs, trg_inputs, e_mask=None, d_mask=None):
        src_inputs = self.src_embedding(src_inputs)  # (B, T_src) -> (B, T_src, d_model)  # 차원추가
        src_inputs = self.positional_encoding(src_inputs)  # 위치 정보 추가

        trg_inputs = self.trg_embedding(trg_inputs)  # (B, T_src) -> (B, T_src, d_model)  # 차원추가
        trg_inputs = self.positional_encoding(trg_inputs)  # 위치 정보 추가

        encoder_outputs = self.encoder(src_inputs, e_mask)  # 소스(원문) 문장 인코딩

        decoder_outputs = self.decoder(trg_inputs, encoder_outputs, e_mask, d_mask)  # 디코딩 (인코더 마스크, 디코더 마스크가 있으면 적용) 결과 출력

        outputs = self.output_layer(decoder_outputs)  # (B, T_trg, d_model) -> (B, T_trg, trg_vocab_size)
        outputs = self.softmax(outputs)  # 각 시점별 타겟 토큰 로그확률 분포

        return outputs  # 최종 로그확률 반환

In [7]:
# PositionalEncoding 구현 (위치정보 추가) : 사인/코사인 함수로 위치정보 만들어 임베딩에 더해줌
class PositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()

        pos_encoding = torch.zeros(seq_len, d_model)  # (T, E)로 위치 인코딩 행렬 생성

        for pos in range(seq_len):
            for i in range(d_model):
                if i % 2 == 0:  # 짝수 차원은 sin함수로 계산 (sin(pos/10000^(2i/d)))
                    pos_encoding[pos, i] = math.sin(pos / (10000 ** (2 * i / d_model)))
                else:  # 홀수 차원은 cos함수로 계산 (cos(pos/10000^(2i/d)))
                    pos_encoding[pos, i] = math.cos(pos / (10000 ** (2 * i / d_model)))

        pos_encoding = pos_encoding.unsqueeze(0)  # (seq_len, d_model) -> (1, seq_len, d_model)

        # pos_encoding은 학습 대상이 아님
        self.pos_encoding = pos_encoding.to(device).requires_grad_(False)

    def forward(self, x):
        x = x * math.sqrt(d_model)  # 임베딩 스케일을 늘려서 위치벡터와 규모를 맞춰줌
        x = x + self.pos_encoding  # (B, seq_len, d_model)에 위치 인코딩 더해줌
        return x  # 위치 정보가 반영된 임베딩 반환

In [9]:
# Transformer FFN(Position-wise Feed Forward) 레이어 구현 : 각 토큰 위치별로 동일한 2층 MLP 적용
class FeedForwardLayer(nn.Module):
    def __init__(self, d_model, d_ff, drop_out_rate):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)  # d_model -> d_ff 확장
        self.relu = nn.ReLU()  # 비선형 추가
        self.dropout = nn.Dropout(drop_out_rate)  # 과적합 완화
        self.linear2 = nn.Linear(d_ff, d_model)  # d_ff -> d_model 축소 (원래 차원으로 축소)

    def forward(self, x):
        x = self.linear1(x)  # (B, T, d_model) -> (B, T, d_ff)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)  # (B, T, d_ff) -> (B, T, d_model)
        return x  # FFN 결과 반환

In [ ]:
# Transformer Layer Normalization 래퍼클래스 구현 : 입력의 마지막 차원(d_model)을 기준으로 정규화(Normalization)해서 학습을 한정화시키는 레이어
class LaterNormalization(nn.Module):
    def __init__(self, )